# Shopee Image Moderation — NSFW

Chỉ detect **NSFW** (khỏa thân, bikini, đồ bó gợi cảm). Violence / QR / logo: implement sau.

Model: [`LukeJacob2023/nsfw-image-detector`](https://huggingface.co/LukeJacob2023/nsfw-image-detector) — ViT 5 lớp `drawings / hentai / neutral / porn / sexy`. Score vi phạm = `sexy + porn + hentai`. Public (không cần login), nhẹ ~340MB.

**Chạy trên server/local:** notebook tự tạo `.venv` ngay trong folder project và cài đặt vào đó — KHÔNG đụng Python hệ thống / ổ C. Model cache cũng để trong project (`.hf_cache/`).

Thứ tự: chạy **cell Bootstrap** → đổi kernel sang *Python (moderation)* → chạy các cell còn lại. Đặt ảnh test vào `sample/`.

## 0. Bootstrap — tạo venv trong folder & cài deps (chạy 1 lần, kernel nào cũng được)

In [ ]:
import os, sys, subprocess, pathlib

PROJ = pathlib.Path.cwd()
if PROJ.name == "notebooks":
    PROJ = PROJ.parent                       # chạy được dù mở từ notebooks/ hay root
VENV = PROJ / ".venv"
PYEXE = VENV / ("Scripts/python.exe" if os.name == "nt" else "bin/python")

if not PYEXE.exists():
    print("Tạo venv:", VENV)
    subprocess.run([sys.executable, "-m", "venv", str(VENV)], check=True)

REQ = ["transformers>=4.50.0", "accelerate", "torch", "pillow<12",
       "pandas", "requests", "ipykernel"]
print("Cài deps vào", VENV, "(lần đầu hơi lâu)...")
subprocess.run([str(PYEXE), "-m", "pip", "install", "-q", "-U", "pip"], check=True)
subprocess.run([str(PYEXE), "-m", "pip", "install", "-q", *REQ], check=True)
subprocess.run([str(PYEXE), "-m", "ipykernel", "install", "--user",
                "--name", "moderation", "--display-name", "Python (moderation)"], check=True)
print("\nXONG. Đổi kernel: Kernel > Change Kernel > 'Python (moderation)', rồi chạy từ cell 1.")
# Lưu ý: server có GPU NVIDIA muốn bản CUDA của torch -> cài tay trong venv:
#   .venv/bin/pip install torch --index-url https://download.pytorch.org/whl/cu121

## 1. Cấu hình (chạy trong kernel *Python (moderation)*)

Đặt cache model vào folder project trước khi import transformers.

In [ ]:
import os, pathlib
PROJ = pathlib.Path.cwd()
if PROJ.name == "notebooks":
    PROJ = PROJ.parent
os.environ["HF_HOME"] = str(PROJ / ".hf_cache")   # weights tải về đây, không vào ~/.cache (C:)
IMAGE_DIR = str(PROJ / "sample")                  # thư mục ảnh test
print("HF_HOME  :", os.environ["HF_HOME"])
print("IMAGE_DIR:", IMAGE_DIR)

## 2. Load model NSFW (5 lớp)

In [ ]:
import torch
from transformers import (pipeline, AutoModelForImageClassification,
                          AutoImageProcessor, ViTImageProcessor)

NSFW_MODEL = "LukeJacob2023/nsfw-image-detector"
_device = 0 if torch.cuda.is_available() else -1

_mdl = AutoModelForImageClassification.from_pretrained(NSFW_MODEL)
try:
    _ip = AutoImageProcessor.from_pretrained(NSFW_MODEL)
except Exception:
    _ip = ViTImageProcessor.from_pretrained(NSFW_MODEL)  # repo thiếu image_processor_type
nsfw_clf = pipeline("image-classification", model=_mdl, image_processor=_ip, device=_device)
print("NSFW labels:", list(_mdl.config.id2label.values()), "| device:", _device)

## 3. Hàm moderation

In [ ]:
from PIL import Image
import requests, io

_HEADERS = {"User-Agent": "Mozilla/5.0"}

def load_image(src):
    if isinstance(src, Image.Image):
        return src.convert("RGB")
    if str(src).startswith("http"):
        r = requests.get(src, headers=_HEADERS, timeout=30); r.raise_for_status()
        return Image.open(io.BytesIO(r.content)).convert("RGB")
    return Image.open(src).convert("RGB")

UNSAFE_LABELS = {"porn", "hentai", "sexy"}   # 'sexy' bắt bikini/đồ bó; bỏ neutral/drawings
ALL_LABELS = ["neutral", "drawings", "sexy", "porn", "hentai"]
NSFW_THR = 0.5        # tinh chỉnh sau khi xem bảng + có ảnh sạch

def nsfw_breakdown(src):
    return {p["label"].lower(): float(p["score"]) for p in nsfw_clf(load_image(src))}

def moderate(src):
    nb = nsfw_breakdown(src)
    nsfw = sum(nb.get(k, 0.0) for k in UNSAFE_LABELS)
    return {**{k: nb.get(k, 0.0) for k in ALL_LABELS},
            "nsfw": nsfw,
            "verdict": "REJECT" if nsfw >= NSFW_THR else "ACCEPT"}

## 4. Test 1 ảnh

In [ ]:
TEST_IMAGE = os.path.join(IMAGE_DIR, "sample-caught-01.jpg")   # đổi path hoặc URL
r = moderate(TEST_IMAGE)
print(r["verdict"], "| nsfw =", round(r["nsfw"], 3))
print({k: round(v, 3) for k, v in r.items() if isinstance(v, float)})

## 5. Quét cả folder ảnh

In đủ điểm 5 lớp + tổng `nsfw` + verdict. Xem cột `sexy`/`neutral` để hiểu lý do và dò
threshold (nên có cả ảnh vi phạm LẪN ảnh sạch trong folder).

In [ ]:
import pandas as pd, glob, time

paths = sorted(p for e in ("*.jpg","*.jpeg","*.png","*.webp","*.bmp")
               for p in glob.glob(os.path.join(IMAGE_DIR, e)))
print(f"Quét {len(paths)} ảnh trong {IMAGE_DIR}\n")

rows, t0 = [], time.time()
for i, p in enumerate(paths, 1):
    r = moderate(p)
    row = {"file": os.path.basename(p)}
    row.update({k: round(r[k], 3) for k in ALL_LABELS})
    row["nsfw"] = round(r["nsfw"], 3)
    row["verdict"] = r["verdict"]
    rows.append(row)
    print(f"[{i}/{len(paths)}] {row['file']:40} nsfw={row['nsfw']:.3f}  {row['verdict']}")

df = pd.DataFrame(rows).sort_values("nsfw", ascending=False)
print(f"\nXong {len(paths)} ảnh / {time.time()-t0:.1f}s | REJECT: {(df['verdict']=='REJECT').sum()}")
df